# 03 — Point Forecast Model: Why ONE Global LightGBM Model

Goal: understand why `train_forecast.py` trains a single model across all 30,490
item-store series, instead of 30,490 separate models — and what the categorical
columns are actually doing to make that possible.

We'll build a tiny version by hand: 3 series, one shared model, and inspect what
it learns.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path

FEATURES_PATH = Path("../data/features.parquet")
df = pd.read_parquet(FEATURES_PATH)
df.shape


(57473650, 22)

## Step 1 — the naive alternative: per-series models

The intuitive first idea: train a separate small model for EACH item-store pair,
since each series might behave differently. Let's see why this doesn't scale,
using real numbers from your data.

In [2]:
n_series = df.groupby(['item_id','store_id']).ngroups
avg_rows_per_series = len(df) / n_series
print(f"Number of distinct series: {n_series:,}")
print(f"Average rows per series: {avg_rows_per_series:.0f}")

Number of distinct series: 30,490
Average rows per series: 1885


Two problems with per-series models, given these numbers:

1. **Data starvation.** Each series only has ~this many rows of history. A
   LightGBM model with dozens of features needs more data than that per series to
   learn anything reliable — especially for low-volume items with lots of zero-sales
   days (common in retail).
2. **Operational cost.** Training, tracking, and serving 30,490 separate models is
   a real MLOps burden — 30,490 model files, 30,490 sets of metrics to monitor for
   drift, 30,490 things that can silently break.

A global model solves both: it pools data across ALL series, so a model can learn
general demand patterns (weekday effects, price sensitivity, holiday spikes) from
the FULL dataset, while still specializing per series via the categorical features.

## Step 2 — how does ONE model still make series-specific predictions?

This is the part that isn't obvious. The trick: `item_id` and `store_id` are
included as INPUT FEATURES (categorical), not just as grouping keys. Let's prove
this by building a tiny model on 3 series and watching it produce different
predictions for the same feature values, purely based on which series it is.

In [7]:
# grab 3 real series to keep this concrete
sample_ids = df[['item_id','store_id']].drop_duplicates().head(3)
sample_ids

,item_id,store_id
0,FOODS_1_001,CA_1
1885,FOODS_1_001,CA_2
3770,FOODS_1_001,CA_3


In [8]:
sample_df = df.merge(sample_ids, on=['item_id','store_id'])
sample_df = sample_df.copy()
for c in ['item_id','store_id']:
    sample_df[c] = sample_df[c].astype('category')

feature_cols = ['item_id','store_id','wday','lag_7']  # tiny feature set for clarity
target_col = 'sales'

train_set = lgb.Dataset(
    sample_df[feature_cols], label=sample_df[target_col],
    categorical_feature=['item_id','store_id']
)
model = lgb.train({'objective':'regression','verbose':-1}, train_set, num_boost_round=50)
print("trained")


trained


In [11]:
# now: same wday and lag_7, but DIFFERENT item_id/store_id -> does the prediction change?
test_rows = pd.DataFrame({
    'item_id': [sample_ids.iloc[0]['item_id'], sample_ids.iloc[1]['item_id'], sample_ids.iloc[2]['item_id']],
    'store_id': [sample_ids.iloc[0]['store_id'], sample_ids.iloc[1]['store_id'], sample_ids.iloc[2]['store_id']],
    'wday': [3, 3, 3],       # SAME for all three
    'lag_7': [2.0, 2.0, 2.0]  # SAME for all three
})
for c in ['item_id','store_id']:
    test_rows[c] = test_rows[c].astype('category')

test_rows['predicted_sales'] = model.predict(test_rows[feature_cols])
test_rows


,item_id,store_id,wday,lag_7,predicted_sales
0,FOODS_1_001,CA_1,3,2.0,0.887141
1,FOODS_1_001,CA_2,3,2.0,1.536267
2,FOODS_1_001,CA_3,3,2.0,1.437802


**This is the key result.** Every input row has the exact same `wday` and `lag_7`
— the only thing that differs is `item_id`/`store_id`. If the model is doing its
job, the predictions should differ across rows too. That difference IS the model
learning series-specific behavior, using ONE set of trees, not 3 separate models.

Mechanically: LightGBM can split a tree on a categorical column just like a numeric
one (e.g. "if item_id is in this group, go left; otherwise go right"). With enough
trees, the model builds up item/store-specific adjustments on top of the general
patterns (weekday effects, lag effects) it learns from the whole dataset.

## Step 3 — feature importance: what is the model actually leaning on?

Let's train on the REAL feature set (not the tiny 4-column version) and look at
which features matter most, using MLflow's logged model or a fresh quick fit.

In [15]:

import sys
sys.path.insert(0, '..')  # so `from src...` resolves when running from notebooks/

import src
from src.models.common import FEATURE_COLS, TARGET_COL, load_features, time_split

full_df = load_features()
train_df, val_df = time_split(full_df)

cat_cols = [c for c in FEATURE_COLS if full_df[c].dtype.name == 'category']
train_set = lgb.Dataset(train_df[FEATURE_COLS], label=train_df[TARGET_COL], categorical_feature=cat_cols)
quick_model = lgb.train({'objective':'regression','verbose':-1}, train_set, num_boost_round=100)

importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'gain': quick_model.feature_importance(importance_type='gain')
}).sort_values('gain', ascending=False)
importance


,feature,gain
16,roll_mean_7,2.687240e+09
18,roll_mean_28,3.152848e+08
7,wday,5.507457e+07
0,item_id,3.441046e+07
15,lag_28,3.208298e+07
14,lag_14,3.205149e+07
13,lag_7,1.809397e+07
12,snap_active,8.754467e+06
3,store_id,7.254108e+06
17,roll_std_7,7.156855e+06


In [23]:
importance.reset_index(drop=True).head(10)

,feature,gain
0,roll_mean_7,2.687240e+09
1,roll_mean_28,3.152848e+08
2,wday,5.507457e+07
3,item_id,3.441046e+07
4,lag_28,3.208298e+07
5,lag_14,3.205149e+07
6,lag_7,1.809397e+07
7,snap_active,8.754467e+06
8,store_id,7.254108e+06
9,roll_std_7,7.156855e+06


Look at where `item_id` and `store_id` rank relative to lag/rolling features.
If they rank highly, that confirms the model is genuinely using series identity to
adjust its predictions — not just relying on generic lag/calendar patterns and
ignoring which series it's looking at.

This table is also literally what gets logged to MLflow as `feature_importance.png`
in the real training run — now you know how to read it, not just that it exists.

## Step 4 — the honest tradeoff

Global models aren't free of downsides. Worth being able to state this out loud:

- A single very high-volume series (say, a top-selling item at a flagship store)
  can't get a model that's PERFECTLY specialized to just its own quirks — it's
  sharing capacity with 30,489 other series.
- If you genuinely needed maximum accuracy on a handful of extremely important
  series, a hybrid approach (global model + a few dedicated per-series models for
  the top movers) is a legitimate answer — worth mentioning as a "what I'd do with
  more time" in an interview, not something you need to build for v1.

## Check your understanding

1. In your own words: how does ONE LightGBM model produce different predictions
   for different series, given the same lag/calendar feature values?
2. Why does a per-series model approach struggle on low-volume items specifically?
3. Where did `item_id`/`store_id` rank in the feature importance table? What does
   that tell you?
4. Name one legitimate downside of the global-model approach.

Next: notebook 04 — the quantile/risk model, what "P90" actually optimizes for,
and the synthetic inventory logic.